# Function Calling/Tool Use


## Just a quick definition

Function calling in the context of Large Language Models (LLMs) refers to the ability of the model to invoke external functions or tools during its response generation. Instead of only generating text, the LLM can recognize when a task requires external data or computation, call a predefined function with the appropriate arguments, and then use the function's output to continue its response.

Think of having a super smart robot that you can only speak with. Pretty useless at doing anything aside from talking. Now imagine you have given that robot a hammer and some nails. It can now put up that wall painting that's been waiting forever to be hung :D

Except, the way we as AI Engineers give AI tools is by defining python functions and describing that function in detail to the LLM so it knows what tools it has access to, what each tool can do, what are its inputs and expected outputs.

Just remember, a *tool* in its simplest form is just a *python function* that you have defined. It can be as simple as a calculator function or something like being able to call external APIs.

![tool-use](../images/tool-use.png)

*Image courtesy of [API Deck](https://www.apideck.com/blog/llm-tool-use-and-function-calling)*

## Step 1: Import libraries

In [1]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import Markdown, display

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception("API key is missing")

## Step 2: Define a tool

In [2]:
import requests

def get_weather(latitude, longitude):
    response = requests.get(f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m,wind_speed_10m&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m")
    data = response.json()
    return data['current']['temperature_2m']

In [3]:
get_weather(24.343627, 54.497922)

22.8

## Step 3: Call a chat model normally

In [4]:
client = OpenAI()

response = client.responses.create(
    model="gpt-4.1-nano",
    input=[
        {"role": "user", "content": "What is the weather like in Abu Dhabi now?"}
    ]
)

print(response.output_text)

I don't have real-time weather data access. To find the current weather in Abu Dhabi, I recommend checking a reliable weather website or app such as Weather.com, AccuWeather, or using a weather service through your device's assistant.


In [5]:
response.__dict__

{'id': 'resp_00397594384dbd550069c30770e95481a39d6367f183faa18a',
 'created_at': 1774389104.0,
 'error': None,
 'incomplete_details': None,
 'instructions': None,
 'metadata': {},
 'model': 'gpt-4.1-nano-2025-04-14',
 'object': 'response',
 'output': [ResponseOutputMessage(id='msg_00397594384dbd550069c30771eb5481a39f909785b2db630f', content=[ResponseOutputText(annotations=[], text="I don't have real-time weather data access. To find the current weather in Abu Dhabi, I recommend checking a reliable weather website or app such as Weather.com, AccuWeather, or using a weather service through your device's assistant.", type='output_text', logprobs=[])], role='assistant', status='completed', type='message')],
 'parallel_tool_calls': True,
 'temperature': 1.0,
 'tool_choice': 'auto',
 'tools': [],
 'top_p': 1.0,
 'background': False,
 'conversation': None,
 'max_output_tokens': None,
 'max_tool_calls': None,
 'previous_response_id': None,
 'prompt': None,
 'prompt_cache_key': None,
 'prompt_c

In [6]:
response.output[0].__dict__

{'id': 'msg_00397594384dbd550069c30771eb5481a39f909785b2db630f',
 'content': [ResponseOutputText(annotations=[], text="I don't have real-time weather data access. To find the current weather in Abu Dhabi, I recommend checking a reliable weather website or app such as Weather.com, AccuWeather, or using a weather service through your device's assistant.", type='output_text', logprobs=[])],
 'role': 'assistant',
 'status': 'completed',
 'type': 'message'}

## Step 4: Define the input schema for our tool

Lets try giving our `get_weather` function as a tool to our AI.

We will first need to define the tool in a format OpenAI can understand

In [7]:
tools = [{
    'type': 'function',
    'name': 'get_weather',
    'description': 'Fetch the current weather for a specific location',
    'parameters': {
        'type': 'object',
        'properties': {
            'latitude': {'type': 'number'},
            'longitude': {'type': 'number'}
        },
        'required': ['latitude', 'longitude'],
        'additionalProperties': False
    },
    'strict': True
}]

## Step 5: Pass the tool schema over to the model

In [8]:
input_messages = [{"role": "user", "content": "What is the weather like in Abu Dhabi now?"}]

In [9]:
response = client.responses.create(
    model="gpt-4.1-nano",
    input=input_messages,
    tools=tools
)

In [10]:
print(response.output_text)

In [11]:
response.__dict__

{'id': 'resp_0020553bee6a35350069c307fdc8c0819db5a94039bf6787e4',
 'created_at': 1774389245.0,
 'error': None,
 'incomplete_details': None,
 'instructions': None,
 'metadata': {},
 'model': 'gpt-4.1-nano-2025-04-14',
 'object': 'response',
 'output': [ResponseFunctionToolCall(arguments='{"latitude":24.4539,"longitude":54.3773}', call_id='call_70lnV5ghHczQlQCWssEspo2D', name='get_weather', type='function_call', id='fc_0020553bee6a35350069c307ff3480819d80b480610611b5cc', status='completed')],
 'parallel_tool_calls': True,
 'temperature': 1.0,
 'tool_choice': 'auto',
 'tools': [FunctionTool(name='get_weather', parameters={'type': 'object', 'properties': {'latitude': {'type': 'number'}, 'longitude': {'type': 'number'}}, 'required': ['latitude', 'longitude'], 'additionalProperties': False}, strict=True, type='function', description='Fetch the current weather for a specific location')],
 'top_p': 1.0,
 'background': False,
 'conversation': None,
 'max_output_tokens': None,
 'max_tool_calls':

In [12]:
response.output[0].__dict__

{'arguments': '{"latitude":24.4539,"longitude":54.3773}',
 'call_id': 'call_70lnV5ghHczQlQCWssEspo2D',
 'name': 'get_weather',
 'type': 'function_call',
 'id': 'fc_0020553bee6a35350069c307ff3480819d80b480610611b5cc',
 'status': 'completed'}

## Step 6: Format the tool call response from the LLM

In [13]:
import json

tool_call = response.output[0]
args = json.loads(tool_call.arguments)

In [14]:
print(tool_call)

ResponseFunctionToolCall(arguments='{"latitude":24.4539,"longitude":54.3773}', call_id='call_70lnV5ghHczQlQCWssEspo2D', name='get_weather', type='function_call', id='fc_0020553bee6a35350069c307ff3480819d80b480610611b5cc', status='completed')


In [15]:
print(args)

{'latitude': 24.4539, 'longitude': 54.3773}


## Step 7: Pass on the tool call arguments to our tool/python function

We now need to pass on the arguments received by the model to our python function or tool

In [16]:
result = get_weather(args['latitude'], args['longitude'])
result

22.8

## Step 8: Append the response of the tool into the message list

In [17]:
input_messages.append(tool_call)

input_messages.append({
    "type": "function_call_output",
    "call_id": tool_call.call_id,
    "output": str(result)
})

In [18]:
from pprint import pprint
pprint(input_messages)

[{'content': 'What is the weather like in Abu Dhabi now?', 'role': 'user'},
 ResponseFunctionToolCall(arguments='{"latitude":24.4539,"longitude":54.3773}', call_id='call_70lnV5ghHczQlQCWssEspo2D', name='get_weather', type='function_call', id='fc_0020553bee6a35350069c307ff3480819d80b480610611b5cc', status='completed'),
 {'call_id': 'call_70lnV5ghHczQlQCWssEspo2D',
  'output': '22.8',
  'type': 'function_call_output'}]


## Step 9: Pass the message list into the model

In [19]:
response_2 = client.responses.create(
    model="gpt-4.1-nano",
    input=input_messages,
    tools=tools
) 

In [20]:
print(response_2.output_text)

The current temperature in Abu Dhabi is approximately 22.8°C.


In [21]:
response_2.__dict__

{'id': 'resp_0020553bee6a35350069c3089901ec819db36eab6cccfc9940',
 'created_at': 1774389401.0,
 'error': None,
 'incomplete_details': None,
 'instructions': None,
 'metadata': {},
 'model': 'gpt-4.1-nano-2025-04-14',
 'object': 'response',
 'output': [ResponseOutputMessage(id='msg_0020553bee6a35350069c30899baf8819d8b33ed0be7480248', content=[ResponseOutputText(annotations=[], text='The current temperature in Abu Dhabi is approximately 22.8°C.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message')],
 'parallel_tool_calls': True,
 'temperature': 1.0,
 'tool_choice': 'auto',
 'tools': [FunctionTool(name='get_weather', parameters={'type': 'object', 'properties': {'latitude': {'type': 'number'}, 'longitude': {'type': 'number'}}, 'required': ['latitude', 'longitude'], 'additionalProperties': False}, strict=True, type='function', description='Fetch the current weather for a specific location')],
 'top_p': 1.0,
 'background': False,
 'conversation': None,
 '

# Resources:

- [OpenAIs Function Calling Guide](https://platform.openai.com/docs/guides/function-calling?api-mode=responses)
- [Anthropics Tool Use Guide with Claude](https://docs.anthropic.com/en/docs/agents-and-tools/tool-use/overview)
- [Function Calling with Gemini API](https://ai.google.dev/gemini-api/docs/function-calling?example=meeting)

<div style="border-radius:16px;background:#1e2a1e;margin:1em 0;padding:1em 1em 1em 3em;color:#eceff4;position:relative;box-shadow:0 6px 16px rgba(0,0,0,.4)">
  <b style="color:#a3be8c;font-size:1.25em">Your Challenge:</b>
  <div style="position:absolute;top:-.8em;left:-.8em;width:2.4em;height:2.4em;border-radius:50%;background:#a3be8c;color:#2e3440;display:flex;align-items:center;justify-content:center;font-weight:700;font-size:1.2em">💪</div>

  <br>
  Now that you've seen how to define a Python function as a tool and connect it to an LLM, it's time to get creative!

  <b>Pick your challenge</b><br>
  <b>1. Wikipedia Article Summarizer</b><br>
  - Fetch a wikipedia article you are interested in learning about and have an LLM summarize it<br>
  - <a href="https://pypi.org/project/Wikipedia-API/">Wikipedia API</a><br>

  <b>2. News Summarizer</b><br>
  - Fetch latest headlines or news by topic.<br>
  - <a href="https://newsapi.org">NewsAPI</a><br>
  - <a href="https://currentsapi.services/en/docs/">CurrentsAPI</a><br>

  <b>3. Stock Market Data</b><br>
  - Get real-time stock prices or company info<br>
  - <a href="https://www.alphavantage.co/">Alpha Vantage</a><br>
  - <a href="https://pypi.org/project/yfinance/">yfinance</a><br>

  <b>4. Movie Info</b><br>
  - Get movie ratings, cast, plot summaries.<br>
  - <a href="https://imdbapi.dev/">IMDb</a><br>
  - <a href="https://www.omdbapi.com/">OMDb API</a><br>

  <b>5. NASA API</b><br>
  - Astronomy facts, country data, etc.<br>
  - <a href="https://api.nasa.gov/">NASA APIs</a><br>

  <b>6. Your own custom tool</b><br>
  - Think of a real-world use case where an LLM could benefit from calling a custom function.<br>

  <hr>

  <b>Tips</b>:<br>
  - Use clear function names and docstrings.<br>
  - Handle input arguments and outputs carefully.<br>
  - Print the LLM's tool call and your function's output in a readable way.<br>

  Be sure to place your submissions in <code>part1-fundamentals/community-contributions/&lt;your-name&gt;</code><br>

  I'm super excited to see what you come up with :D
</div>

## Step 1. Import libraries

In [56]:
import os
import yfinance as yf
from openai import OpenAI
from dotenv import load_dotenv
from pprint import pprint

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception("API key is missing")

client = OpenAI()

## Step 2. Define the tool (yfinance)

In [57]:
from datetime import datetime, timedelta

def get_stock_data(ticker):
    """
    Fetch yesterday's stock data (open, high, low, close, volume)
    for a given ticker symbol.
    """
    stock = yf.Ticker(ticker)

    # Több napot kérünk, hogy biztos legyen adat (hétvége miatt)
    data = stock.history(period="5d")

    if data.empty:
        return "No data available for this ticker."

    # Utolsó elérhető nap (nem feltétlen tegnap → hétvége kezelve)
    last_row = data.iloc[-1]

    return {
        "date": str(data.index[-1].date()),
        "open": round(float(last_row["Open"]), 2),
        "high": round(float(last_row["High"]), 2),
        "low": round(float(last_row["Low"]), 2),
        "close": round(float(last_row["Close"]), 2),
        "volume": int(last_row["Volume"])
    }

## Step 3. Tool schema (LLM)

In [58]:
tools = [{
    "type": "function",
    "name": "get_stock_data",
    "description": "Fetch the latest available daily stock data (open, high, low, close, volume) for a given ticker symbol",
    "parameters": {
        "type": "object",
        "properties": {
            "ticker": {
                "type": "string",
                "description": "The stock ticker symbol, e.g. AAPL, TSLA"
            }
        },
        "required": ["ticker"],
        "additionalProperties": False
    },
    "strict": True
}]

## Step 4. Input messages

In [ ]:
from pprint import pprint

company = "TESLA"
input_messages = [
    {"role": "system", "content": "You are a financial assistant. Use tools when needed."},
    {"role": "user", "content": f"Provide the latest stock data for {company}."}
]

response = client.responses.create(
    model="gpt-4.1-nano",
    input=input_messages,
    tools=tools
)

pprint(response)

Response(id='resp_0745d0cf29ad77150069caf573d5bc81a18d348b2b4235990d', created_at=1774908787.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-4.1-nano-2025-04-14', object='response', output=[ResponseFunctionToolCall(arguments='{"ticker":"BRK-B"}', call_id='call_p4ZENgEOx0wvtzr5iG8noPph', name='get_stock_data', type='function_call', id='fc_0745d0cf29ad77150069caf574ba6881a1b914a62f0a28c2b8', namespace=None, status='completed')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[FunctionTool(name='get_stock_data', parameters={'type': 'object', 'properties': {'ticker': {'type': 'string', 'description': 'The stock ticker symbol, e.g. AAPL, TSLA'}}, 'required': ['ticker'], 'additionalProperties': False}, strict=True, type='function', defer_loading=None, description='Fetch the latest available daily stock data (open, high, low, close, volume) for a given ticker symbol')], top_p=1.0, background=False, completed_at=1774908788.0, conversatio

## Step 5. Call our tool

In [52]:
import json

tool_call = response.output[0]

if tool_call.type == "function_call":
    args = json.loads(tool_call.arguments)
    print("Tool call:", tool_call)
    print("Arguments:", args)
else:
    print("No tool call detected")

Tool call: ResponseFunctionToolCall(arguments='{"ticker":"BRK-B"}', call_id='call_p4ZENgEOx0wvtzr5iG8noPph', name='get_stock_data', type='function_call', id='fc_0745d0cf29ad77150069caf574ba6881a1b914a62f0a28c2b8', namespace=None, status='completed')
Arguments: {'ticker': 'BRK-B'}


## Step 6. Run the tool

In [53]:
result = get_stock_data(args["ticker"])
print("Stock data:", result)

Stock data: {'date': '2026-03-30', 'open': 470.68, 'high': 477.69, 'low': 470.73, 'close': 474.66, 'volume': 3015482}


## Step 7. Tool answer

In [54]:
input_messages.append(tool_call)

input_messages.append({
    "type": "function_call_output",
    "call_id": tool_call.call_id,
    "output": str(result)
})

pprint(input_messages)

[{'content': 'You are a financial assistant. Use tools when needed.',
  'role': 'system'},
 {'content': 'Provide the latest stock data for Berkshire Hathaway BRK-B.',
  'role': 'user'},
 ResponseFunctionToolCall(arguments='{"ticker":"BRK-B"}', call_id='call_p4ZENgEOx0wvtzr5iG8noPph', name='get_stock_data', type='function_call', id='fc_0745d0cf29ad77150069caf574ba6881a1b914a62f0a28c2b8', namespace=None, status='completed'),
 {'call_id': 'call_p4ZENgEOx0wvtzr5iG8noPph',
  'output': "{'date': '2026-03-30', 'open': 470.68, 'high': 477.69, 'low': "
            "470.73, 'close': 474.66, 'volume': 3015482}",
  'type': 'function_call_output'}]


## Step 8. Second model call (final response)

In [55]:
response_2 = client.responses.create(
    model="gpt-4.1-nano",
    input=input_messages,
    tools=tools
)

print("\nStock Summary:\n")
print(response_2.output_text)


Stock Summary:

The latest stock data for Berkshire Hathaway (BRK-B) is as follows:
- Date: March 30, 2026
- Opening Price: $470.68
- Highest Price: $477.69
- Lowest Price: $470.73
- Closing Price: $474.66
- Volume: 3,015,482 shares
